# STEP 5: Baselines, Stage 1 Screening & Architecture Evaluation

This notebook presents the verified empirical evaluation for Step 5 on the **TRAIN partition** ($N=220,953$ rows, 2023-11-01 to 2025-12-07):
1. **Purged Walk-Forward Cross-Validation** (6 expanding folds, 12-candle purge, 132,523 held-out evaluation candles).
2. **Canonical Baselines Suite**:
   - **Binary Directional Baselines** (15 rules across $N \in \{2, 3, 6, 12\}$).
   - **Three-Class Baselines** (Balanced accuracy mathematically anchored at 33.33%).
   - **Volatility Baseline** scored strictly on `big_move_N`.
3. **Like-for-Like Multi-Coverage Comparison** (100%, 50%, 20%, 10% coverage) with stationary block-bootstrap 95% CIs ($B=288$).
4. **20% Coverage Gap Decomposition**: Direction Skill vs. Size Selection Hurdle ($gap = (hit - 0.5) - (BE - 0.5)$).
5. **Stage 1 Screening & Survivors**: 76 Logistic Regression configurations, ranking, and directional hit rates.

> **Strict Partition Guard:** Evaluated strictly on the `train` partition. Validation rows remain reserved for Step 6, and test rows remain locked.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ML_DIR = Path.cwd()
if (ML_DIR / 'ML').exists():
    ML_DIR = ML_DIR / 'ML'
elif ML_DIR.name == 'notebooks':
    ML_DIR = ML_DIR.parent
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

from src import config
print(f'Working Directory: {ML_DIR}')
print(f'Candidate Horizons: {config.CANDIDATE_HORIZONS}')
print(f'Round-Trip Cost: {config.TOTAL_ROUNDTRIP_COST*10000:.1f} bps ({config.TOTAL_ROUNDTRIP_COST})')

Working Directory: C:\B.TECH\Academics\SEMESTER V\03 Machine Learning\.ML LABs\Project\Stock-Market-ML-Indicator\ML
Candidate Horizons: [2, 3, 6, 12]
Round-Trip Cost: 23.0 bps (0.0023)


## 1. Purged Walk-Forward Fold Boundaries

Evaluated on train post-warmup ($N=218,935$ candles, 760.2 calendar days). Purge gap is strictly 12 candles.

In [2]:
cv_folds_path = ML_DIR / 'results' / 'reports' / 'cv_folds_summary.csv'
df_folds = pd.read_csv(cv_folds_path)
display_cols = [
    'fold', 'train_rows', 'purge_rows', 'held_out_rows',
    'train_start_time', 'train_end_time',
    'held_out_start_time', 'held_out_end_time'
]
print('=== Purged Walk-Forward 6-Fold Structure ===')
print(df_folds[display_cols].to_string(index=False))
print(f'\nTotal evaluated held-out candles: {df_folds["held_out_rows"].sum():,}')
print(f'Purge gap strictly enforced: {df_folds["purge_rows"].iloc[0]} candles')

=== Purged Walk-Forward 6-Fold Structure ===
 fold  train_rows  purge_rows  held_out_rows          train_start_time            train_end_time       held_out_start_time         held_out_end_time
    1       86400          12          22088 2023-11-08 00:10:00+00:00 2024-09-03 00:05:00+00:00 2024-09-03 01:10:00+00:00 2024-11-18 17:45:00+00:00
    2      108488          12          22087 2023-11-08 00:10:00+00:00 2024-11-18 16:45:00+00:00 2024-11-18 17:50:00+00:00 2025-02-03 10:20:00+00:00
    3      130575          12          22087 2023-11-08 00:10:00+00:00 2025-02-03 09:20:00+00:00 2025-02-03 10:25:00+00:00 2025-04-21 02:55:00+00:00
    4      152662          12          22087 2023-11-08 00:10:00+00:00 2025-04-21 01:55:00+00:00 2025-04-21 03:00:00+00:00 2025-07-06 19:30:00+00:00
    5      174749          12          22087 2023-11-08 00:10:00+00:00 2025-07-06 18:30:00+00:00 2025-07-06 19:35:00+00:00 2025-09-21 12:05:00+00:00
    6      196836          12          22087 2023-11-08 00:10

## 2. Benchmark Baselines Evaluation

### 2A. Binary Classification Baselines (`raw_sign_N` / `bin_dir_N`)

Directional forecasting across 15 rules. RSI(14) uses true abstentions (<30 Long, >70 Short, Neutral dropped); EMA50 emits both signs cleanly.

In [3]:
bin_path = ML_DIR / 'results' / 'reports' / 'baselines_binary_summary.csv'
df_bin = pd.read_csv(bin_path)
disp_cols_bin = [
    'baseline', 'horizon', 'accuracy', 'balanced_accuracy',
    'f1_macro', 'mcc', 'coverage', 'signals_per_day', 'hit_rate',
    'break_even_rate', 'break_even_gap', 'net_return_bps'
]
print('=== Binary Baselines Evaluation (Horizon N=12 / 60-Minute Forward Move) ===')
df_bin_12 = df_bin[df_bin['horizon'] == 12].sort_values(by='break_even_gap', ascending=False).reset_index(drop=True)
print(df_bin_12[disp_cols_bin].to_string(index=False))

# Dynamic takeaways
top_bin = df_bin_12.iloc[0]
rsi_bin = df_bin_12[df_bin_12['baseline'] == 'rsi_14'].iloc[0]
ema_bin = df_bin_12[df_bin_12['baseline'] == 'ema50_rule'].iloc[0]
loc_bin = df_bin_12[df_bin_12['baseline'] == 'loc_position_288'].iloc[0]
print('\n--- Dynamic Analysis (Binary Directional Regime) ---')
print(f'1. Top Directional Baseline: {top_bin["baseline"]} (Hit Rate: {top_bin["hit_rate"]*100:.2f}%, Gap: {top_bin["break_even_gap"]:.4f}, Net: {top_bin["net_return_bps"]:.2f} bps)')
print(f'2. RSI(14) Abstentions Preserved: Coverage = {rsi_bin["coverage"]*100:.2f}% ({rsi_bin["signals_per_day"]:.1f} sig/day), Hit Rate = {rsi_bin["hit_rate"]*100:.2f}%, MCC = {rsi_bin["mcc"]:.4f}')
print(f'3. EMA50 Directional Rule: Coverage = {ema_bin["coverage"]*100:.2f}%, Hit Rate = {ema_bin["hit_rate"]*100:.2f}%, MCC = {ema_bin["mcc"]:.4f}')
print(f'4. Location 288 Mean Reversion: Hit Rate = {loc_bin["hit_rate"]*100:.2f}%, Gap = {loc_bin["break_even_gap"]:.4f}, Net = {loc_bin["net_return_bps"]:.2f} bps')

=== Binary Baselines Evaluation (Horizon N=12 / 60-Minute Forward Move) ===
            baseline  horizon  accuracy  balanced_accuracy  f1_macro       mcc  coverage  signals_per_day  hit_rate  break_even_rate  break_even_gap  net_return_bps
              rsi_14       12  0.541277           0.541424  0.541276  0.082852  0.071773        20.670647  0.541277         0.781889       -0.240612      -22.308049
      reverse_rsi_14       12  0.458723           0.458576  0.458574 -0.082852  0.071773        20.670647  0.458723         0.781889       -0.323166      -23.691951
     ref_5feature_lr       12  0.522572           0.520988  0.516403  0.042936  1.000000       288.000000  0.522572         0.864951       -0.342379      -22.818241
reverse_rule_vote_4b       12  0.520973           0.521236  0.520940  0.042489  0.375358       108.102984  0.520973         0.863580       -0.342607      -22.737804
    loc_position_288       12  0.518266           0.519058  0.517301  0.038321  0.999977       287.

### 2B. Three-Class Classification Baselines (`target_N_S1`)

Evaluated on 3-class target `[-1.0, 0.0, 1.0]`. Balanced accuracy (unweighted macro recall) mathematically anchors all constant and random baselines at the exact **33.33%** zero-skill watermark despite ~52.7% Neutral prevalence.

In [4]:
three_path = ML_DIR / 'results' / 'reports' / 'baselines_three_class_summary.csv'
df_three = pd.read_csv(three_path)
disp_cols_3 = [
    'baseline', 'horizon', 'accuracy', 'balanced_accuracy',
    'precision_macro', 'recall_macro', 'f1_macro', 'mcc',
    'coverage', 'signals_per_day', 'hit_rate', 'break_even_gap', 'net_return_bps'
]
print('=== Three-Class Baselines Evaluation (Horizon N=12, Rule S1) ===')
df_three_12 = df_three[df_three['horizon'] == 12].sort_values(by='balanced_accuracy', ascending=False).reset_index(drop=True)
print(df_three_12[disp_cols_3].to_string(index=False))

neutral_b = df_three_12[df_three_12['baseline'] == 'always_neutral'].iloc[0]
rand3_b = df_three_12[df_three_12['baseline'] == 'random_3class'].iloc[0]
up3_b = df_three_12[df_three_12['baseline'] == 'always_up'].iloc[0]
print('\n--- Dynamic Analysis (Three-Class Balanced Accuracy Watermark) ---')
print(f'1. Always-Neutral: Raw Accuracy is {neutral_b["accuracy"]*100:.2f}%, but Balanced Accuracy is exactly {neutral_b["balanced_accuracy"]*100:.2f}% (Macro F1: {neutral_b["f1_macro"]:.4f})')
print(f'2. Random 3-Class: Raw Accuracy is {rand3_b["accuracy"]*100:.2f}%, Balanced Accuracy is {rand3_b["balanced_accuracy"]*100:.2f}%, MCC is {rand3_b["mcc"]:.4f}')
print(f'3. Zero-Skill Invariance: Always-Up ({up3_b["balanced_accuracy"]*100:.2f}%), Always-Down, Coin-Flip, and Majority Class all collapse to strictly 33.33% balanced accuracy.')

=== Three-Class Baselines Evaluation (Horizon N=12, Rule S1) ===
            baseline  horizon  accuracy  balanced_accuracy  precision_macro  recall_macro  f1_macro       mcc  coverage  signals_per_day  hit_rate  break_even_gap  net_return_bps
          ema50_rule       12  0.236961           0.344818         0.158385      0.344818  0.216843  0.014789  1.000000       288.000000  0.516131       -0.348887      -23.006813
              rsi_14       12  0.530907           0.344231         0.378129      0.344231  0.283970  0.040534  0.071760        20.666903  0.541277       -0.240612      -22.308049
    reverse_momentum       12  0.237029           0.343228         0.413647      0.343228  0.217621  0.013016  0.997578       287.302337  0.509955       -0.354657      -22.524261
    loc_position_288       12  0.235377           0.342996         0.491366      0.342996  0.215593  0.012631  0.999977       287.993480  0.518172       -0.346839      -22.965388
      reverse_rsi_14       12  0.528915 

### 2C. Volatility Baseline on `big_move_N`

`volatility_lr` is evaluated strictly as a non-directional magnitude model predicting whether $|fwd\_ret\_N| > 23$ bps.

In [5]:
vol_path = ML_DIR / 'results' / 'reports' / 'volatility_baseline_summary.csv'
df_vol = pd.read_csv(vol_path)
print('=== Volatility Baseline Evaluation (Predicting big_move_N via feat_vol_sigma_288) ===')
print(df_vol.to_string(index=False))

vol_12 = df_vol[df_vol['horizon'] == 12].iloc[0]
print(f'\nAt N=12: ROC AUC = {vol_12["roc_auc"]:.4f}, PR AUC = {vol_12["pr_auc"]:.4f}, F1 = {vol_12["f1_score"]:.4f}, Brier = {vol_12["brier_score"]:.4f}')

=== Volatility Baseline Evaluation (Predicting big_move_N via feat_vol_sigma_288) ===
        model            feature      target  horizon  horizon_min  roc_auc   pr_auc  f1_score  brier_score  positive_prevalence
volatility_lr feat_vol_sigma_288  big_move_2        2           10 0.705666 0.293555  0.063156     0.122907             0.154383
volatility_lr feat_vol_sigma_288  big_move_3        3           15 0.687743 0.357286  0.126869     0.155800             0.210700
volatility_lr feat_vol_sigma_288  big_move_6        6           30 0.667133 0.481741  0.228472     0.206609             0.330939
volatility_lr feat_vol_sigma_288 big_move_12       12           60 0.647375 0.589266  0.465733     0.233876             0.458234

At N=12: ROC AUC = 0.6474, PR AUC = 0.5893, F1 = 0.4657, Brier = 0.2339


## 3. Like-for-Like Multi-Coverage Comparison (100%, 50%, 20%, 10%)

Evaluated at fixed coverage thresholds established on training folds. Primary metric: Directional Hit Rate vs `raw_sign_12` with stationary block-bootstrap 95% CIs ($B=288$ candles), mean $|fwd\_ret\_open\_12|$, break-even rate, gap, and net bps.

In [6]:
lfl_path = ML_DIR / 'results' / 'reports' / 'like_for_like_summary.csv'
df_lfl = pd.read_csv(lfl_path)

for cov in [1.0, 0.5, 0.2, 0.1]:
    print(f'\n================ Coverage Level: {int(cov*100)}% ================')
    sub = df_lfl[df_lfl['target_coverage'] == cov].sort_values(by='break_even_gap', ascending=False).reset_index(drop=True)
    sub_cols = ['model', 'signals_per_day', 'hit_rate', 'ci_lower', 'ci_upper', 'mean_abs_fwd_ret', 'break_even_rate', 'break_even_gap', 'net_bps']
    print(sub[sub_cols].to_string(index=False))


================ Coverage Level: 100% ================
               model  signals_per_day  hit_rate  ci_lower  ci_upper  mean_abs_fwd_ret  break_even_rate  break_even_gap    net_bps
            C_12_F15       131.792006  0.522690  0.514391  0.531135          0.004393         0.761765       -0.239075 -21.006359
              rsi_14        20.665032  0.541277  0.518982  0.564205          0.004079         0.781899       -0.240622 -19.632251
            C_12_F10       228.417165  0.522406  0.515774  0.528602          0.003576         0.821628       -0.299222 -21.397724
      reverse_rsi_14        20.665032  0.458723  0.435795  0.481018          0.004079         0.781899       -0.323175 -26.367749
            C_12_F05       286.943821  0.523626  0.517991  0.528853          0.003159         0.864085       -0.340459 -21.507499
                B_12       287.921765  0.523372  0.517144  0.528339          0.003151         0.864953       -0.341580 -21.527046
     ref_5feature_lr       287.921

## 4. 20% Coverage Gap Decomposition: Direction Skill vs. Size Selection Hurdle

Decomposes break-even gap into:
$$\text{gap} = (\text{hit} - 0.5) - (\text{BE} - 0.5)$$
- **Direction Skill:** $\text{hit} - 0.5$
- **Size Selection Hurdle:** $\text{BE} - 0.5 = \frac{c}{2 \cdot \mathbb{E}[|fwd\_ret\_open|]}$

In [7]:
decomp_path = ML_DIR / 'results' / 'reports' / 'like_for_like_decomp_20pct.csv'
df_decomp = pd.read_csv(decomp_path)
df_decomp_sorted = df_decomp.sort_values(by='break_even_gap', ascending=False).reset_index(drop=True)
print('=== 20% Coverage Gap Decomposition ===')
disp_decomp = ['model', 'hit_rate', 'direction_skill', 'mean_abs_fwd_ret', 'break_even_rate', 'size_selection_hurdle', 'break_even_gap', 'net_bps']
print(df_decomp_sorted[disp_decomp].to_string(index=False))

loc_row = df_decomp[df_decomp['model'] == 'loc_position_288'].iloc[0]
vol_row = df_decomp[df_decomp['model'] == 'vol_gated_reversal'].iloc[0]
c15_row = df_decomp[df_decomp['model'] == 'C_12_F15'].iloc[0]
b12_row = df_decomp[df_decomp['model'] == 'B_12'].iloc[0]

print('\n--- Critical Findings from 20% Coverage Decomposition ---')
print(f'1. Highest Pure Direction Skill: loc_position_288 ({loc_row["hit_rate"]*100:.2f}%, skill: +{loc_row["direction_skill"]*100:.2f}%)')
print(f'2. Highest Fallback ML Direction Skill: B_12 ({b12_row["hit_rate"]*100:.2f}%, skill: +{b12_row["direction_skill"]*100:.2f}%)')
print(f'3. Setup C Direction Skill: C_12_F15 ({c15_row["hit_rate"]*100:.2f}%, skill: +{c15_row["direction_skill"]*100:.2f}%)')
print(f'4. Lowest Break-Even Hurdle: vol_gated_reversal (BE: {vol_row["break_even_rate"]*100:.2f}%, hurdle: +{vol_row["size_selection_hurdle"]*100:.2f}%, mean |fwd_ret|: {vol_row["mean_abs_fwd_ret"]*100:.3f}%)')
print(f'5. Tightest Break-Even Gap: vol_gated_reversal ({vol_row["break_even_gap"]:.4f}) beats C_12_F15 ({c15_row["break_even_gap"]:.4f}) and B_12 ({b12_row["break_even_gap"]:.4f})!')

=== 20% Coverage Gap Decomposition ===
               model  hit_rate  direction_skill  mean_abs_fwd_ret  break_even_rate  size_selection_hurdle  break_even_gap    net_bps
  vol_gated_reversal  0.518330         0.018330          0.006039         0.690424               0.190424       -0.172094 -20.786089
            C_12_F15  0.527438         0.027438          0.005571         0.706425               0.206425       -0.178987 -19.942818
            C_12_F10  0.533007         0.033007          0.005285         0.717598               0.217598       -0.184590 -19.511125
          ema50_rule  0.534977         0.034977          0.004973         0.731267               0.231267       -0.196290 -19.521447
            C_12_F05  0.538283         0.038283          0.004759         0.741641               0.241641       -0.203358 -19.356126
    reverse_momentum  0.516013         0.016013          0.004867         0.736300               0.236300       -0.220288 -21.441428
              rsi_14  0.541277

## 5. Stage 1 Screening: 76 Logistic Regression Configurations

Screening of all 76 configurations on 70 `LINEAR_FEATURES`. Direction hit rate shown next to rank metric.

In [8]:
stage1_path = ML_DIR / 'results' / 'reports' / 'stage1_lr_ranking.csv'
df_stage1 = pd.read_csv(stage1_path)
s_cols = [
    'config_id', 'setup', 'horizon', 'rule', 'signals_per_day',
    'mean_hit_rate', 'mean_break_even_gap', 'std_break_even_gap',
    'rank_metric', 'mean_net_bps', 'status'
]
survivors = df_stage1[df_stage1['status'].isin(['survivor', 'fallback'])].copy()
print('=== Stage 1 Survivors & Fallback ===')
print(survivors[s_cols].to_string(index=False))

b12_s = survivors[survivors['config_id'] == 'B_12'].iloc[0]
c15_s = survivors[survivors['config_id'] == 'C_12_F15'].iloc[0]
print(f'\nNotice: B_12 has the HIGHEST direction hit rate ({b12_s["mean_hit_rate"]*100:.2f}%) among all survivors (vs C_12_F15 at {c15_s["mean_hit_rate"]*100:.2f}%).')
print('Setup C\'s superior rank metric is driven by size selection (volatility conditioning), not direction skill.')

=== Stage 1 Survivors & Fallback ===


config_id   setup  horizon     rule  signals_per_day  mean_hit_rate  mean_break_even_gap  std_break_even_gap  rank_metric  mean_net_bps   status
 C_12_F15 setup_c       12      F15        28.782010       0.528970            -0.175858            0.038144    -0.214002    -21.269923 survivor
 C_12_F10 setup_c       12      F10        48.050375       0.535283            -0.198893            0.040747    -0.239639    -21.003567 survivor
 C_12_F05 setup_c       12      F05        58.017930       0.538663            -0.225862            0.051633    -0.277495    -21.083109 survivor
     B_12 setup_b       12 raw_sign        66.626857       0.546112            -0.291696            0.084368    -0.376064    -21.119889 fallback

Notice: B_12 has the HIGHEST direction hit rate (54.61%) among all survivors (vs C_12_F15 at 52.90%).
Setup C's superior rank metric is driven by size selection (volatility conditioning), not direction skill.
